# 🎬 HunyuanVideo (13B) - Ejecución en Google Colab con GPU Gratuita

Este cuaderno permite ejecutar **Tencent HunyuanVideo (13B)** en Google Colab aprovechando GPUs dedicadas (T4 o A100) sin gastar hardware local.

---

### Paso 1: Comprobar GPU de Colab

In [ ]:
!nvidia-smi

### Paso 2: Instalar ComfyUI y extensiones para HunyuanVideo

In [ ]:
# Clonar ComfyUI y gestor de paquetes
!git clone https://github.com/comfyanonymous/ComfyUI
%cd ComfyUI
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt

# Instalar nodo especializado para HunyuanVideo (Kijai)
%cd custom_nodes
!git clone https://github.com/kijai/ComfyUI-HunyuanVideoWrapper
%cd ComfyUI-HunyuanVideoWrapper
!pip install -r requirements.txt
%cd /content/ComfyUI

### Paso 3: Descargar pesos optimizados (FP8 / GGUF) de HunyuanVideo

In [ ]:
# Descargar pesos cuantizados para máxima velocidad en Colab T4/A100
!wget -c https://huggingface.co/Kijai/HunyuanVideo_comfy/resolve/main/hunyuan_video_720_fp8_e4m3fn.safetensors -P models/checkpoints/
!wget -c https://huggingface.co/Kijai/HunyuanVideo_comfy/resolve/main/hunyuan_video_vae_bf16.safetensors -P models/vae/
!wget -c https://huggingface.co/Kijai/HunyuanVideo_comfy/resolve/main/llava_llama3_fp8_scaled.safetensors -P models/clip/

### Paso 4: Iniciar ComfyUI y generar enlace público con Cloudflare

In [ ]:
# Iniciar túnel Cloudflare seguro para abrir la interfaz en tu navegador
!wget -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

import subprocess
import threading
import time
import socket

def run_comfy():
    !python main.py --listen 127.0.0.1 --port 8188 --highvram

threading.Thread(target=run_comfy, daemon=True).start()
time.sleep(5)

!cloudflared tunnel --url http://127.0.0.1:8188